## 0 · 环境自检

In [ ]:
import importlib
for m in ['numpy','pandas','matplotlib']:
    mod = importlib.import_module(m)
    print('OK ', m, mod.__version__)

## 1 · `import`：把工具箱搬进来

`as np` 是取别名，少打字。`from X import Y` 只拿一个东西。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 自动找 data 文件夹
CANDS = [
    '../../workshop_build/workshop_build/09_advanced_ml/gan/data',
    'workshop_build/workshop_build/09_advanced_ml/gan/data',
    'data', '../data', '../../gan/data',
]
DATA = None
for guess in CANDS:
    if os.path.exists(os.path.join(guess, 'stakeholder_relations.parquet')):
        DATA = guess; break
print('文件夹：', DATA)

df = pd.read_parquet(os.path.join(DATA, 'stakeholder_relations.parquet'))
print('数据形状：', df.shape, '(列 = 街区, 栏 = 特征)')
df.head()

In [ ]:
# 让 matplotlib 显示中文（找不到字体就略过，不影响程序）
from matplotlib import font_manager as fm
for fp in [
    '../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    '../../../.venv/Lib/site-packages/matplotlib/mpl-data/fonts/ttf/NotoSansCJKsc-Regular.otf',
    'C:/Windows/Fonts/msjh.ttc',
]:
    if os.path.exists(fp):
        try:
            fm.fontManager.addfont(fp)
            plt.rcParams['font.sans-serif'] = [fm.FontProperties(fname=fp).get_name()]
            break
        except Exception: pass
plt.rcParams['axes.unicode_minus'] = False

**拆解**
- `import numpy as np` / `pandas as pd` / `matplotlib.pyplot as plt`：加载并取别名。
- `os.path.join(a,b)`：安全接路径。`for...if os.path.exists` 找到文件夹就 `break`。
- `pd.read_parquet(路径)`：把档读成表格 `df`。`df.shape`=(列,栏)。`df.head()` 看前 5 列。

## 2 · 看懂表格 `DataFrame`

In [ ]:
print('共', len(df.columns), '栏：')
print(list(df.columns))
print('---')
print('每栏类型：'); print(df.dtypes)

## 3 · list 推导式：一行挑出要的字段

```python
feat_cols = [c for c in df.columns if c not in drop and np.issubdtype(df[c].dtype, np.number)]
```
读法：「对每个栏名 c，若不在 drop 里且是数字栏，就留下」。

In [ ]:
# 先看最小例子
nums = [1,2,3,4,5]
print('平方：', [n*n for n in nums])
print('偶数：', [n for n in nums if n % 2 == 0])

In [ ]:
drop = ['site','gx','gy']
feat_cols = [c for c in df.columns
             if c not in drop and np.issubdtype(df[c].dtype, np.number)]
print('特征栏 %d 个：' % len(feat_cols)); print(feat_cols)

X = df[feat_cols].to_numpy(dtype='float32')   # 表格 → 纯数字数组
print('X 形状：', X.shape)

## 4 · `numpy` 数组与标准化

numpy 让你「对整个数组一次运算」。标准化：每栏变平均 0、标准差 1。
```python
mu = X.mean(axis=0)   # 每栏平均
sd = X.std(axis=0)    # 每栏标准差
Xs = (X - mu) / sd    # 广播：整表一次处理
```

In [ ]:
mu = X.mean(axis=0)
sd = X.std(axis=0); sd[sd < 1e-8] = 1.0   # 避免除以 0
Xs = (X - mu) / sd
print('标准化后：', Xs.shape)
print('每栏平均≈0：', Xs.mean(axis=0).round(2)[:5], '...')
print('每栏标准差≈1：', Xs.std(axis=0).round(2)[:5], '...')

## 5 · 布尔遮罩与 `argmax`：图上颜色哪来的

- `argmax(axis=1)`：每列找最大值的位置（0/1/2/3）。
- `dominant == i`：产生一排 True/False，用来只挑某一类。

In [ ]:
share_cols = ['share_state','share_developer','share_resident','share_unknown']
share_idx  = [feat_cols.index(c) for c in share_cols]
names  = ['state 政府','developer 开发商','resident 居民','unknown 未知']
colors = ['#1f77b4','#d62728','#2ca02c','#7f7f7f']

dominant = X[:, share_idx].argmax(axis=1)
print('前 10 个街区主导者：', dominant[:10])
print('state 主导的街区数：', (dominant == 0).sum())

## 6 · `def` 函数：把步骤打包

`def 名字(参数):` → 缩进内容 → `return` 交回结果。

In [ ]:
def standardize(arr):
    m = arr.mean(axis=0); s = arr.std(axis=0); s[s < 1e-8] = 1.0
    return (arr - m) / s

print('和上面一样吗：', np.allclose(Xs, standardize(X)))

## 7 · `matplotlib` 画点图

套路：**for 循环，一类一个颜色，叠在同一张图**。这里先用最简单的 PCA 压成 2 维来画。

In [ ]:
from sklearn.decomposition import PCA
Z = PCA(n_components=2).fit_transform(Xs)   # (5438, 2)

plt.figure(figsize=(8,6))
for i in range(4):
    m = (dominant == i)
    plt.scatter(Z[m,0], Z[m,1], s=8, c=colors[i], label=names[i], alpha=0.6)
plt.xlabel('维度 1'); plt.ylabel('维度 2')
plt.title('5438 个街区压成 2 维')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**拆解（三本都一样）**：`plt.figure()` 开图 → `for i in range(4)` 四类 → `plt.scatter(x,y,c=,label=)` 画点 → `plt.legend()`/`plt.show()`。改 `s=`、`alpha=` 看效果。

---
## 第 1 本完成
你会了：`import` / `pandas` 读表 / list 推导式 / `numpy` 标准化 / 布尔遮罩 / `def` / `matplotlib`。

**下一步**：打开 **`ML_教学.ipynb`**，学 `class` 和 `torch` 训练循环。